# Strain conditioning and whitening: numerical controls

These examples fix input bytes, precision, sample spacing, and calculation
options. They compare the original CPU pipeline, the default JAX pipeline,
and original-kernel validation controls. Default JAX conditioning runs on the
selected device. Validation controls transfer selected stages to the CPU and
are deliberately slow.

Set `PYCBC_NOTEBOOK_JAX_DEVICE` to a supported device, such as `cuda:0`, to
repeat the comparisons there. Results describe the displayed libraries and
device; they are not a promise of identical rounding across installations.

In [1]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

import jax
import numpy as np

from pycbc import scheme
from pycbc.fft import backend_cpu
from pycbc.frame import write_frame
from pycbc.strain import detect_loud_glitches, gate_data
from pycbc.strain.strain import StrainBuffer
from pycbc.types import TimeSeries
from pycbc.types.array_jax import to_jax

DEVICE = os.environ.get("PYCBC_NOTEBOOK_JAX_DEVICE", "cpu")
with scheme.JAXScheme(DEVICE) as context:
    print({"NumPy": np.__version__, "JAX": jax.__version__,
           "CPU FFT": backend_cpu.cpu_backend,
           "JAX device": context.jax_device.device_kind})


def compare(expected, actual):
    assert expected.shape == actual.shape
    assert expected.dtype == actual.dtype
    difference = actual - expected
    scale = np.linalg.norm(expected)
    return {"different values": int(np.count_nonzero(actual != expected)),
            "maximum absolute difference": float(np.max(np.abs(difference))),
            "relative L2 difference": float(np.linalg.norm(difference) / scale)
            if scale else float(np.linalg.norm(difference))}


def exact(expected, actual):
    assert expected.dtype == actual.dtype
    assert expected.shape == actual.shape
    assert expected.tobytes() == actual.tobytes()

{'NumPy': '2.5.2', 'JAX': '0.11.1', 'CPU FFT': 'fftw', 'JAX device': 'cpu'}


## Rounding before cancellation

The original padded whitening stages first multiply the forward FFT by
`delta_t`, divide by the PSD, and then multiply the unnormalized inverse FFT
by `delta_f`. The fused JAX expression cancels the transform normalization
factors algebraically before the inverse transform. The expressions agree in
exact arithmetic. Rounding an intermediate single-precision multiplication
before division can change the result.

This scalar control uses the same float32 operands for both operation orders,
with no FFT library involved. It isolates rounding order rather than
attributing a complete whitening difference to one FFT bin.

In [2]:
value = np.float32(0.86845714)
delta_t = np.float32(0.2)
psd_value = np.float32(1.142317)
scaled_then_divided = (value * delta_t) / psd_value
divided_then_scaled = (value / psd_value) * delta_t
assert scaled_then_divided.tobytes() != divided_then_scaled.tobytes()
print({"scale then divide": repr(scaled_then_divided),
       "divide then scale": repr(divided_then_scaled)})

{'scale then divide': 'np.float32(0.15205187)', 'divide then scale': 'np.float32(0.15205185)'}


## A complete public strain-buffer workflow

A generated float64 GWF contains twelve seconds of deterministic strain-like
noise. The original and JAX paths read the same frame and advance the same
public `StrainBuffer` twice. Conditioning includes a highpass FIR, dynamic
range scaling, float32 storage, LDAS downsampling, and the initial taper.
Whitening includes Welch estimation, both truncated PSD grids, the padded
transform, PSD division, trimming, and endpoint gates.

The native controls select the original implementations at every differing
boundary in this example. Intermediate arrays and the final spectrum must
match CPU dtypes and bytes exactly. The default comparison reports differences
without assuming that every device produces a mismatch.

In [3]:
LIVE_REFERENCES = (
    "firwin", "fir_zero_filter", "resample", "welch", "interpolate",
    "inverse_spectrum_truncation", "fft", "ifft", "divide", "gate_data",
)


def run_buffer(frame_path, *, cpu=False, references=()):
    context = (scheme.CPUScheme() if cpu else
               scheme.JAXScheme(DEVICE, reference_operations=references))
    with context:
        buffer = StrainBuffer(
            [str(frame_path)], "H1:STRAIN", 1187007104,
            max_buffer=8, sample_rate=64, highpass_frequency=5,
            highpass_reduction=30, highpass_bandwidth=4,
            psd_samples=3, psd_segment_length=1, psd_inverse_length=0.5,
            trim_padding=0.25, low_frequency_cutoff=5,
            autogating_taper=0.125, force_update_cache=False)
        for _ in range(2):
            assert buffer.advance(2)
        spectrum = buffer.overwhitened_data(0.5)
        series = {"conditioned strain": buffer.strain, "Welch PSD": buffer.psd,
                  "padded PSD": spectrum.psd.psdt,
                  "output PSD": spectrum.psd, "whitened spectrum": spectrum}
        if not cpu:
            assert all(to_jax(value).devices() == {context.jax_device}
                       for value in series.values())
        return ({name: value.numpy().copy() for name, value in series.items()},
                {name: (value.dtype.name, str(value.start_time),
                        value.delta_t if isinstance(value, TimeSeries)
                        else value.delta_f) for name, value in series.items()})


samples = (np.random.default_rng(87).normal(size=12 * 128) * 1e-21)
with TemporaryDirectory() as directory:
    frame_path = Path(directory) / "strain.gwf"
    with scheme.CPUScheme():
        write_frame(str(frame_path), "H1:STRAIN", TimeSeries(
            samples, delta_t=1 / 128, epoch=1187007104))
    original, original_metadata = run_buffer(frame_path, cpu=True)
    default, default_metadata = run_buffer(frame_path)
    native, native_metadata = run_buffer(frame_path, references=LIVE_REFERENCES)

assert default_metadata == native_metadata == original_metadata
for name in original:
    exact(original[name], native[name])
    assert np.all(np.isfinite(original[name]))
    print(name, compare(original[name], default[name]))
print("All selected original stages and output metadata match exactly.")

conditioned strain {'different values': 231, 'maximum absolute difference': 2.384185791015625e-07, 'relative L2 difference': 2.3786833480698988e-07}
Welch PSD {'different values': 32, 'maximum absolute difference': 5.587935447692871e-09, 'relative L2 difference': 4.502032879827311e-07}
padded PSD {'different values': 85, 'maximum absolute difference': 3.337860107421875e-06, 'relative L2 difference': 2.9439661375363357e-06}
output PSD {'different values': 53, 'maximum absolute difference': 4.291534423828125e-06, 'relative L2 difference': 3.5162768199370475e-06}
whitened spectrum {'different values': 64, 'maximum absolute difference': 6.869103526696563e-05, 'relative L2 difference': 4.909319386570132e-07}
All selected original stages and output metadata match exactly.


## Tukey gates and threshold decisions

Tukey tapers evaluate cosine values and multiply the data in place. Different
cosine implementations and intermediate rounding can change taper samples.
`gate_data` replaces this complete operation with the original CPU function.
The slice example also checks that gating updates its parent time series.

Glitch detection estimates and truncates a PSD, whitens strain, thresholds its
magnitude, and applies FindChirp clustering. A small magnitude difference can
change a strict threshold decision, so matching peak counts is weaker evidence
than matching intermediate arrays. `detect_loud_glitches` selects the complete
original calculation. `findchirp_cluster` selects just original clustering;
the PSD and whitening controls above isolate the preceding stages.

In [4]:
gates = [(108.0, 0.5, 1.25), (109.25, 0.25, 0.75)]


def gated(*, cpu=False, references=()):
    context = (scheme.CPUScheme() if cpu else
               scheme.JAXScheme(DEVICE, reference_operations=references))
    with context:
        parent = TimeSeries(np.ones(256, np.float64), delta_t=1 / 16, epoch=100)
        view = parent[32:224]
        assert gate_data(view, gates) is view
        np.testing.assert_array_equal(parent.numpy()[32:224], view.numpy())
        return parent.numpy().copy()


original_gates = gated(cpu=True)
default_gates = gated()
native_gates = gated(references=("gate_data",))
exact(original_gates, native_gates)
print("Tukey gates", compare(original_gates, default_gates))

Tukey gates {'different values': 10, 'maximum absolute difference': 2.220446049250313e-16, 'relative L2 difference': 2.232883742297147e-17}


In [5]:
glitch_samples = np.random.default_rng(24).normal(size=512).astype(np.float32)
glitch_samples[250:253] += 70
options = dict(psd_duration=1, psd_stride=0.5, low_freq_cutoff=5,
               corrupt_time=0.125, threshold=5, cluster_window=0.25)


def glitch_times(*, cpu=False, references=()):
    context = (scheme.CPUScheme() if cpu else
               scheme.JAXScheme(DEVICE, reference_operations=references))
    with context:
        strain = TimeSeries(glitch_samples, delta_t=1 / 64, epoch=1187007104)
        return np.asarray(detect_loud_glitches(strain, **options), dtype=float)


original_times = glitch_times(cpu=True)
default_times = glitch_times()
native_times = glitch_times(references=("detect_loud_glitches",))
exact(original_times, native_times)
assert len(original_times)
print({"original peak times": original_times.tolist(),
       "default peak times": default_times.tolist(),
       "original-control peak times": native_times.tolist()})

{'original peak times': [1187007107.9375], 'default peak times': [1187007107.9375], 'original-control peak times': [1187007107.9375]}


## Scope of the controls

Select operations independently with
`scheme.JAXScheme(DEVICE, reference_operations=("fft",))`, or combine their
names. A selected boundary runs the installed original implementation;
other stages remain JAX. Fused paths use their staged form when required to
reach that boundary. Changing validation options invalidates cached JAX
spectra and prepared PSD grids, so an earlier default result is not reused.
Recreate the buffer or recalculate its base PSD when changing the Welch
implementation; whitening retains a PSD already supplied as an input.

FFT grouping, complex division, Welch reductions, and PSD truncation have
additional controlled examples in the array, FFT, PSD, and filtering
notebooks. The exact native assertions here apply to this configured CPU
backend and these fixed inputs. They do not establish equal search decisions
for an entire observation or equality between different CPU installations.